In [1]:
import cv2
import torch
import torch.nn as nn
from torchvision.models import resnet18, ResNet18_Weights
import torchvision.transforms as T
import numpy as np

# ==========================================
# ۱. کلاس مدل شبکه عصبی
# ==========================================
class AccidentBiLSTM(nn.Module):
    def __init__(self, input_size=512, hidden_size=256, num_layers=1, dropout=0.0):
        super(AccidentBiLSTM, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, bidirectional=True)
        self.fc1 = nn.Linear(hidden_size * 2, 64)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.fc2 = nn.Linear(64, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        pooled_out, _ = torch.max(lstm_out, dim=1) 
        x = self.dropout(self.relu(self.fc1(pooled_out)))
        return self.sigmoid(self.fc2(x)).squeeze(-1)

# ==========================================
# ۲. لود کردن هوشمند مدل‌ها
# ==========================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("در حال بارگذاری مدل‌ها...")

saved_weights = torch.load('best_final_bilstm.pth', map_location=device, weights_only=True)
detected_hidden = saved_weights['fc1.weight'].shape[1] // 2
detected_layers = sum(1 for k in saved_weights.keys() if k.startswith('lstm.weight_ih_l') and 'reverse' not in k)

bilstm_model = AccidentBiLSTM(input_size=512, hidden_size=detected_hidden, num_layers=detected_layers).to(device)
bilstm_model.load_state_dict(saved_weights)
bilstm_model.eval()

resnet = resnet18(weights=ResNet18_Weights.DEFAULT)
resnet = nn.Sequential(*list(resnet.children())[:-1]).to(device)
resnet.eval()

# ==========================================
# ۳. پردازشگر اصلی: هیت‌مپ + ردپای نئونی
# ==========================================
def create_neon_heatmap_demo(video_path, output_name="demo_result.mp4", threshold=0.50):
    print(f"\n🔥 شروع ساخت دموی نئونی-حرارتی برای: {video_path}")
    cap = cv2.VideoCapture(video_path)
    
    raw_frames = []
    frames_tensor = []
    
    # استخراج فریم‌ها
    while True:
        ret, frame = cap.read()
        if not ret: break
        raw_frames.append(frame)
        
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame_t = torch.from_numpy(rgb_frame).permute(2, 0, 1).float() / 255.0
        frame_t = T.Resize((224, 224), antialias=True)(frame_t)
        frames_tensor.append(frame_t)
    cap.release()
    
    if not frames_tensor: return
    
    # پیش‌بینی مدل
    with torch.no_grad():
        video_batch = torch.stack(frames_tensor).to(device)
        features = [resnet(video_batch[i:i+32]).squeeze(-1).squeeze(-1) for i in range(0, len(video_batch), 32)]
        video_features = torch.cat(features, dim=0)
        
        seq_len = video_features.size(0)
        if seq_len < 100:
            padding = torch.zeros(100 - seq_len, 512).to(device)
            video_features = torch.cat((video_features, padding), dim=0)
        else:
            video_features = video_features[:100, :]
            
        prob = bilstm_model(video_features.unsqueeze(0)).item()
        
    is_accident = prob >= threshold
    print(f"📊 درصد اطمینان هوش مصنوعی: {prob*100:.1f}%")

    # پیدا کردن فریم دقیق برخورد
    impact_idx = len(raw_frames) # در حالت عادی، هیچوقت به این نمی‌رسد
    if is_accident:
        diffs = [np.sum(cv2.absdiff(cv2.cvtColor(raw_frames[i-1], cv2.COLOR_BGR2GRAY), 
                                    cv2.cvtColor(raw_frames[i], cv2.COLOR_BGR2GRAY))) 
                 for i in range(1, len(raw_frames))]
        impact_idx = np.argmax(diffs) + 1

    H, W = raw_frames[0].shape[:2]
    out = cv2.VideoWriter(output_name, cv2.VideoWriter_fourcc(*'mp4v'), 10, (W, H)) # 10 fps برای نمایش بهتر ردپاها
    
    # لایه‌ی جمع‌کننده (Accumulator) برای افکتِ کش آمدنِ نور (نئون)
    trail_layer = np.zeros((H, W, 3), dtype=np.float32)

    for i in range(1, len(raw_frames)):
        curr = raw_frames[i]
        prev = raw_frames[i-1]
        
        # ۱. محاسبه مناطق دارای حرکت (Motion Mask)
        diff = cv2.absdiff(curr, prev)
        gray_diff = cv2.cvtColor(diff, cv2.COLOR_BGR2GRAY)
        _, mask = cv2.threshold(gray_diff, 20, 255, cv2.THRESH_BINARY)
        mask = cv2.GaussianBlur(mask, (15, 15), 0) # نرم کردن لبه‌های حرکت
        
        color_mask = np.zeros_like(curr, dtype=np.float32)
        
        # ۲. تولید رنگ نئون و هیت‌مپ
        if is_accident and i >= impact_idx:
            # حالت تصادف: هیت‌مپ آتشین (نقشه حرارتی JET: از آبی به قرمز)
            heatmap = cv2.applyColorMap(mask, cv2.COLORMAP_JET)
            # فقط قسمت‌هایی که حرکت داشتند را هیت‌مپ می‌کنیم
            color_mask = cv2.bitwise_and(heatmap, heatmap, mask=mask).astype(np.float32)
            status_text = "STATUS: CRASH DETECTED!"
            status_color = (0, 0, 255) # قرمز
        else:
            # حالت عادی: ردپای نئونی فیروزه‌ای
            color_mask[:, :, 0] = mask # کانال Blue
            color_mask[:, :, 1] = mask # کانال Green
            color_mask[:, :, 2] = 0    # کانال Red (صفر)
            status_text = "STATUS: NORMAL SCAN"
            status_color = (255, 255, 0) # فیروزه‌ای
            
        # ۳. ترکیب با لایه قبلی برای ایجاد افکت کش‌آمدن (Ghosting)
        # 0.7 یعنی ردپاهای قدیمی آرام آرام محو می‌شوند
        trail_layer = trail_layer * 0.7 + color_mask * 0.6
        trail_layer_clipped = np.clip(trail_layer, 0, 255).astype(np.uint8)
        
        # ۴. ترکیب نهایی با ویدیوی اصلی (کمی تاریک کردن پس‌زمینه برای درخشش نئون)
        dark_bg = cv2.addWeighted(curr, 0.6, np.zeros_like(curr), 0.4, 0)
        final_frame = cv2.add(dark_bg, trail_layer_clipped)
        
        # نوشتن اطلاعات روی تصویر
        cv2.putText(final_frame, status_text, (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.8, status_color, 2)
        cv2.putText(final_frame, f"AI Confidence: {prob*100:.1f}%", (20, 80), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)
        
        # کند کردن صحنه تصادف برای جذابیت بیشتر
        times_to_write = 4 if (is_accident and i >= impact_idx and i < impact_idx + 10) else 1
        for _ in range(times_to_write):
            out.write(final_frame)

    out.release()
    print(f"✅ ویدیوی نهایی ساخته شد: {output_name}")

# ==========================================
# ۴. اجرای کد روی ویدیوها
# ==========================================
create_neon_heatmap_demo('./accident_demo_1.mp4', output_name="HEATMAP_CRASH_DEMO.mp4")
create_neon_heatmap_demo('./normal_demo_1.mp4', output_name="HEATMAP_NORMAL_DEMO.mp4")

در حال بارگذاری مدل‌ها...

🔥 شروع ساخت دموی نئونی-حرارتی برای: ./accident_demo_1.mp4
📊 درصد اطمینان هوش مصنوعی: 74.8%
✅ ویدیوی نهایی ساخته شد: HEATMAP_CRASH_DEMO.mp4

🔥 شروع ساخت دموی نئونی-حرارتی برای: ./normal_demo_1.mp4
📊 درصد اطمینان هوش مصنوعی: 43.7%
✅ ویدیوی نهایی ساخته شد: HEATMAP_NORMAL_DEMO.mp4


### هدف

این سلول یک ویدئوی دمو می‌سازد با افکت بصری متفاوت از سلول قبل.

در این نسخه، از افکت نئون فیروزه‌ای و هیت‌مپ آتشین استفاده می‌شود.

هدف: نمایش بصری واضح‌تر و جذاب‌تر برای ارائه.

### بخش ۱: مدل شبکه عصبی

کلاس

`AccidentBiLSTM`

همان مدل Try #4 است.

دوطرفه

`bidirectional=True`

ساختار: LSTM دوطرفه، لایه Linear، ReLU، لایه Linear نهایی، Sigmoid.

### بخش ۲: لود کردن مدل

از فایل وزن ذخیره‌شده استفاده می‌شود.

`best_final_bilstm.pth`

ابعاد مدل از خود وزن‌ها استخراج می‌شوند.

`detected_hidden`

`detected_layers`

مزیت: کد خودکار ابعاد را تشخیص می‌دهد، نیازی به مقدار دستی نیست.

ResNet18

برای استخراج ویژگی هر فریم، بدون لایه آخر.

### بخش ۳: تفاوت بانت بوک heat_neon_nly.ipynb

نت بوک قبل روی هر فریم جدید پیش‌بینی انجام می‌داد.

در این سلول، فقط یک پیش‌بینی برای کل ویدئو انجام می‌شود.

منطق: احتمال تصادف برای کل کلیپ محاسبه می‌شود، نه برای هر لحظه.

نتیجه: اجرای سریع‌تر.

### بخش ۴: پیدا کردن لحظه برخورد

برای اینکه افکت هیت‌مپ در جای درست اعمال شود، باید لحظه‌ی برخورد را بدانیم.

روش: اختلاف بین فریم‌های متوالی محاسبه می‌شود.

`cv2.absdiff`

فریمی که بیشترین اختلاف را دارد، به عنوان لحظه برخورد در نظر گرفته می‌شود.

`impact_idx`

این روش ساده است، ولی برای دمو کافی است.

### بخش ۵: تولید افکت نئون

گام اول: اختلاف بین فریم فعلی و قبلی محاسبه می‌شود.

گام دوم: تبدیل به سیاه و سفید.

گام سوم:

`cv2.threshold`

با مقدار

`20`

گام چهارم: بلور برای نرم کردن لبه‌ها.

اندازه بلور

`۱۵ در ۱۵`

نتیجه: یک ماسک نرم از نواحی حرکت.

### بخش ۶: دو حالت رنگی

اگر تصادف تشخیص داده نشده یا هنوز به لحظه برخورد نرسیده‌ایم:

رنگ نئون فیروزه‌ای.

اعمال از طریق کانال‌های آبی و سبز.

اگر تصادف تشخیص داده شده و از لحظه برخورد گذشته‌ایم:

هیت‌مپ آتشین.

`COLORMAP_JET`

از آبی به قرمز.

### بخش ۷: افکت ردپای نئون

یک لایه تجمعی به نام

`trail_layer`

نگه داشته می‌شود.

هر فریم:

مقدار قبلی ضرب در

`0.7`

می‌شود.

مقدار جدید ضرب در

`0.6`

اضافه می‌شود.

نتیجه: ردپای حرکات با تأخیر محو می‌شود.

این همان افکت "کش آمدن نور" است که در عکس‌های نوردهی طولانی می‌بینیم.

### بخش ۸: تاریک کردن پس‌زمینه

قبل از افزودن افکت، پس‌زمینه کمی تاریک می‌شود.

`cv2.addWeighted`

مقدار: ۶۰ درصد تصویر اصلی، ۴۰ درصد مشکی.

چرا؟ چون افکت نئون روی پس‌زمینه تاریک بیشتر می‌درخشد.

### بخش ۹: متن روی تصویر

دو متن نمایش داده می‌شود.

سمت چپ بالا:

`STATUS: CRASH DETECTED!`

یا

`STATUS: NORMAL SCAN`

زیرش:

`AI Confidence: XX.X%`

### نکته: کند کردن لحظه تصادف

در لحظه برخورد (۱۰ فریم اول بعد از آن)، هر فریم ۴ بار در خروجی نوشته می‌شود.

نتیجه: ویدئو در آن لحظه ۴ برابر کندتر پخش می‌شود.

هدف: بیننده بتواند با دقت لحظه‌ی تشخیص مدل را ببیند.

### خروجی نهایی

دو فایل ساخته می‌شود.

`HEATMAP_CRASH_DEMO.mp4`

`HEATMAP_NORMAL_DEMO.mp4`

این دو فایل در پوشه‌ی

`new_vision`

هستند و به عنوان نمونه‌ی افکت نئون نگه داشته می‌شوند.

### جمع‌بندی

این سلول یک نسخه‌ی جایگزین از ویدئوی دمو می‌سازد.